# Piper TTS

A refresher on **Piper** — a fast, **local** neural text-to-speech system that sounds good and is small enough to run in real time on a **Raspberry Pi 4 (CPU only)**. Each "voice" is a [VITS](https://arxiv.org/abs/2106.06103) model exported to a single **ONNX** file plus a JSON config; you run it with `onnxruntime`, no GPU and no cloud. Piper is the TTS engine behind **Home Assistant**'s voice assistant and the **Rhasspy** project, and the default when you want offline speech on modest hardware.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the phoneme→id contract and the 16-bit PCM output format run on CPU with no download; the real `PiperVoice` synthesis is gated behind `RUN_PIPER`_

## 1. What & Why

**What it is.** Piper is a neural TTS *engine* (originally by Michael Hansen / Rhasspy, now under OHF-Voice) plus a large library of pretrained **voices**. A voice is a [VITS](https://arxiv.org/abs/2106.06103) model — an end-to-end text-to-waveform network — that has been exported to **ONNX**. You drive it from the CLI (`piper -m en_US-lessac-medium -f out.wav`) or from Python (`PiperVoice.load(...).synthesize_wav(...)`). The actual inference is just `onnxruntime` running the `.onnx` graph, so it has tiny dependencies and runs anywhere ONNX runs.

**The problem it solves.** Most good TTS is either a *cloud API* (per-character billing, network latency, your audio leaves the box) or a *heavy PyTorch stack* (Coqui/XTTS, Tortoise — slow on CPU, GPU-hungry). Piper occupies the gap: **near-real-time, fully offline, self-hosted speech on commodity/low-power hardware**, with quality good enough for an assistant or screen reader. A medium voice is ~60 MB and synthesizes faster than real time on a Pi 4.

**When to reach for it.** Offline/edge voice assistants (Home Assistant, Rhasspy); embedded and low-power devices (Raspberry Pi, kiosks); accessibility / screen readers that must work without a network; any app where you want zero per-use cost and audio that never leaves the machine; **80+ languages** with hundreds of community voices already trained.

**When not to.** You need **zero-shot voice cloning** or expressive, emotion-controlled narration — reach for **XTTS-v2 / StyleTTS2 / ElevenLabs**. You want the absolute highest naturalness and don't mind a cloud bill — **ElevenLabs / Azure / Google**. You need rich **SSML** (prosody, phoneme overrides, `<break>`): Piper's control surface is deliberately minimal. Piper trades expressiveness and cloning for speed, size, and being genuinely offline.

## 2. Mental Model

**A Piper voice is a single VITS network frozen into one `.onnx` file. Text becomes *phonemes* (via espeak-ng), phonemes become *integer ids*, the ONNX graph turns those ids directly into a 16-bit waveform. There is no separate vocoder and nothing to train at run time — you're just running one feed-forward graph.**

```
  text            espeak-ng            phoneme→id           VITS  (one .onnx)        16-bit PCM
 ┌────────┐   phonemize (IPA)    ┌──────────────────┐   ┌──────────────────────┐   ┌──────────┐
 │"Hello."│ ─▶ h ə l ˈoʊ ─────▶ │ map + interleave │ ─▶│ encoder→flow→decoder │ ─▶│ 22.05kHz │
 └────────┘    (per-lang rules)  │ pad, add BOS/EOS │   │ (end-to-end, no      │   │ mono wav │
                                 └──────────────────┘   │  separate vocoder)   │   └──────────┘
                                          ▲             └──────────────────────┘
                              phoneme_id_map lives in the voice's .onnx.json config
```

Three ideas make it click:

1. **One file, one graph.** Unlike Coqui's acoustic-model-plus-vocoder pairing, a Piper voice is a *single* end-to-end VITS exported to ONNX. Inference is `session.run(...)` — fast, deterministic-ish, dependency-light.
2. **The `.onnx.json` config is the contract.** Alongside every `voice.onnx` sits `voice.onnx.json` holding the **sample rate**, the **espeak voice** to phonemize with, the **`phoneme_id_map`** (IPA symbol → integer id), the number of speakers, and the **default inference scales** (`length_scale`, `noise_scale`, `noise_w`). The model is useless without it.
3. **Quality is a tier baked into the voice name.** `<lang>_<REGION>-<name>-<quality>` (e.g. `en_US-lessac-medium`). Quality ∈ `x_low, low, medium, high` sets model size and sample rate (`x_low`/`low` = 16 kHz, `medium`/`high` = 22.05 kHz). You pick the speed/quality trade-off by *choosing a voice*, not by tuning at run time.

## 3. Key Concepts

- **Voice = `.onnx` + `.onnx.json`.** Two files always travel together: the ONNX weights and the JSON config (sample rate, espeak voice, `phoneme_id_map`, `num_speakers`, default scales). Download both or synthesis fails.
- **VITS, end-to-end.** Each voice is a VITS model: a text encoder, a normalizing-flow duration/latent model, and a HiFi-GAN-style decoder, trained jointly. **No separate vocoder** — phoneme ids go in, a waveform comes out.
- **espeak-ng phonemization.** Piper converts text to **IPA phonemes** using `espeak-ng` (the config names the language voice, e.g. `en-us`). This is why pronunciation is language-rule-driven, not learned from your raw text — and why espeak-ng (or its bundled data) must be present.
- **`phoneme_id_map` + special tokens.** The config maps each IPA symbol to integer id(s). Piper frames the sequence with **BOS `^`**, **EOS `$`**, and interleaves a **PAD `_`** between every phoneme — the model was trained on that exact layout. Get the framing wrong and you get garbage.
- **Quality tiers.** `x_low` (16 kHz, ~5–7M params), `low` (16 kHz), `medium` (22.05 kHz, the common default), `high` (22.05 kHz, largest). Higher tier = better audio, more compute, larger file.
- **Inference scales.** `length_scale` controls **speed** (>1 slower/longer, <1 faster); `noise_scale` controls latent variability (expressiveness vs stability); `noise_w` controls **phoneme-duration** variability (cadence). Defaults live in the config; override per call.
- **Multi-speaker voices.** Some voices (e.g. `en_US-libritts_r-medium`) bundle many speakers; pass a `speaker_id` (0..`num_speakers-1`). Single-speaker voices ignore it.
- **Output format.** **16-bit signed PCM, mono**, at the voice's sample rate. Streaming synthesis emits audio **sentence by sentence**, so long text starts playing before it finishes.

## 4. Setup

```bash
# Python engine + CLI (imports as `piper`):
pip install piper-tts

# Download a voice (weights + config) into the current dir:
python -m piper.download_voices en_US-lessac-medium
#   -> en_US-lessac-medium.onnx  and  en_US-lessac-medium.onnx.json

# Synthesize from the CLI:
echo "Piper runs entirely on the CPU." | piper -m en_US-lessac-medium -f out.wav
```

`espeak-ng` is required for phonemization. The `piper-tts` wheels bundle a `espeak-ng` build,
but on minimal systems install the system package too (`apt-get install espeak-ng` /
`brew install espeak-ng`). Voices are hosted on Hugging Face under
[`rhasspy/piper-voices`](https://huggingface.co/rhasspy/piper-voices); a `medium` voice is
~60 MB.

> **Note on packages.** In 2025 Piper was relicensed/rewritten as **piper1-gpl** with a
> cleaner Python API (`PiperVoice.load(...).synthesize_wav(...)`); `pip install piper-tts`
> installs it. Older guides show `voice.synthesize_stream_raw(...)` — same engine, older API.

To stay self-contained and download-free, the runnable cells below reproduce Piper's
**phoneme→id contract** (Example 1) and its **16-bit PCM WAV output format** (Example 2) with
only the standard library + `numpy`. The real `PiperVoice` synthesis (Example 3) is shown but
gated behind `RUN_PIPER`, so the notebook always executes.

In [ ]:
import sys
import wave
import numpy as np

print(f"python {sys.version.split()[0]}, numpy {np.__version__}")
print("Piper pipeline = text -> espeak-ng phonemes -> phoneme ids -> VITS(.onnx) -> 16-bit PCM wav")
print("Next two cells reproduce the id contract and the output format, no model download.")

## 5. Worked Examples

### Example 1 — Phonemes → ids, exactly the way Piper frames them

A Piper voice's `.onnx.json` carries a `phoneme_id_map`: each IPA symbol maps to integer
id(s). The VITS model was trained on a **specific framing** — start with **BOS `^`**, then for
every phoneme emit its id(s) **followed by a PAD `_`**, and finish with **EOS `$`**. That
interleaved-pad layout is not optional: feed the model a bare id sequence and the audio falls
apart. Below we reproduce `phonemes_to_ids` with a tiny stand-in config (real maps have
~150 entries) so the contract is concrete and dependency-free.

In [ ]:
# A miniature stand-in for the voice config's "phoneme_id_map".
# Real Piper configs map ~150 IPA symbols; values are LISTS (a symbol can map to >1 id).
PAD, BOS, EOS = "_", "^", "$"
phoneme_id_map = {
    PAD: [0], BOS: [1], EOS: [2], " ": [3],
    "h": [10], "\u0259": [11], "l": [12], "o": [13], "\u028a": [14],
    "\u02c8": [15], "w": [16], "\u0259r": [17], "d": [18], ".": [19],
}

def phonemes_to_ids(phonemes):
    """Frame phonemes the way Piper's VITS expects: BOS, then each phoneme + PAD, then EOS."""
    ids = list(phoneme_id_map[BOS])
    for p in phonemes:
        if p not in phoneme_id_map:
            continue                      # Piper silently drops unknown symbols
        ids.extend(phoneme_id_map[p])
        ids.extend(phoneme_id_map[PAD])   # <-- the critical interleaved pad
    ids.extend(phoneme_id_map[EOS])
    return ids

# espeak-ng would produce these IPA phonemes for "Hello world." — we hard-code its output here.
phonemes = ["h", "\u0259", "l", "o", "\u028a", " ", "w", "\u02c8", "\u0259r", "l", "d", "."]
ids = phonemes_to_ids(phonemes)

print(f"phonemes ({len(phonemes)}): {phonemes}")
print(f"ids      ({len(ids)}): {ids}")
print(f"framing  : starts with BOS={phoneme_id_map[BOS]}, ends with EOS={phoneme_id_map[EOS]}")
print(f"pad count: {ids.count(0)}  (one PAD after every kept phoneme)")
print("This int sequence is exactly what gets fed to session.run() on the .onnx graph.")

### Example 2 — Piper's output format: 16-bit mono PCM at the voice's sample rate

VITS emits a float waveform in `[-1, 1]`; Piper writes it as **16-bit signed PCM, mono**, at
the voice's sample rate (22.05 kHz for a `medium` voice). Knowing this format is what lets you
stream it, concatenate sentences, or pipe it to `aplay`/`ffmpeg`. Here we synthesize a stand-in
tone (no model needed), convert it exactly as Piper does, write a real WAV with the stdlib
`wave` module, and read the header back to confirm the contract.

In [ ]:
SAMPLE_RATE = 22050          # a "medium" Piper voice; "x_low"/"low" would be 16000

# Stand in for VITS output: 0.4 s, two-tone "ah" in float [-1, 1].
dur = 0.4
t = np.linspace(0, dur, int(SAMPLE_RATE * dur), endpoint=False)
audio_float = (0.6 * np.sin(2 * np.pi * 180 * t)
               + 0.3 * np.sin(2 * np.pi * 360 * t)).astype(np.float32)
audio_float *= np.hanning(audio_float.size).astype(np.float32)   # fade edges

# Exactly Piper's float->int16 conversion: clip, scale by 32767, cast.
audio_int16 = (np.clip(audio_float, -1.0, 1.0) * 32767).astype(np.int16)

out_path = "/tmp/piper_demo.wav"
with wave.open(out_path, "wb") as wav_file:
    wav_file.setnchannels(1)          # mono
    wav_file.setsampwidth(2)          # 16-bit -> 2 bytes/sample
    wav_file.setframerate(SAMPLE_RATE)
    wav_file.writeframes(audio_int16.tobytes())

# Read the header back to prove the format.
with wave.open(out_path, "rb") as r:
    ch, width, sr, n = r.getnchannels(), r.getsampwidth(), r.getframerate(), r.getnframes()

print(f"wrote {out_path}")
print(f"channels={ch}  sampwidth={width*8}-bit  rate={sr} Hz  frames={n}  ({n/sr:.2f}s)")
print(f"peak amplitude: {np.abs(audio_int16).max()} / 32767")
print("Same bytes Piper's synthesize_wav() would write — ready for aplay/ffmpeg/concat.")

### Example 3 — The real engine: `PiperVoice` synthesis (gated)

With `piper-tts` installed and a voice downloaded, synthesis is a few lines. The voice
download is ~60 MB, so this is gated behind `RUN_PIPER=1`. Either way the cell prints the
canonical call shapes — Python (`synthesize_wav`, with `SynthesisConfig` to control
speed/expressiveness and pick a `speaker_id`) and the equivalent CLI.

In [ ]:
import os

if os.getenv("RUN_PIPER"):
    import wave
    from piper import PiperVoice, SynthesisConfig

    # Assumes you ran: python -m piper.download_voices en_US-lessac-medium
    voice = PiperVoice.load("en_US-lessac-medium.onnx")   # loads .onnx + .onnx.json

    syn = SynthesisConfig(length_scale=1.0, noise_scale=0.667, noise_w_scale=0.8)
    with wave.open("out.wav", "wb") as wav_file:
        voice.synthesize_wav("Piper runs entirely on the CPU.", wav_file, syn_config=syn)
    print("wrote out.wav")
else:
    print("Set RUN_PIPER=1 after `pip install piper-tts` and downloading a voice.\n")
    print("# 1) fetch a voice (weights + config) -> en_US-lessac-medium.onnx(.json)")
    print("python -m piper.download_voices en_US-lessac-medium\n")
    print("# 2) Python API")
    print("from piper import PiperVoice, SynthesisConfig")
    print("import wave")
    print('voice = PiperVoice.load("en_US-lessac-medium.onnx")')
    print("syn = SynthesisConfig(length_scale=1.0,   # >1 slower, <1 faster")
    print("                      noise_scale=0.667,  # expressiveness")
    print("                      noise_w_scale=0.8,  # cadence variability")
    print("                      speaker_id=None)    # set for multi-speaker voices")
    print('with wave.open("out.wav", "wb") as f:')
    print('    voice.synthesize_wav("Hello there.", f, syn_config=syn)\n')
    print("# 3) equivalent CLI (synthesizes sentence-by-sentence, streaming)")
    print('echo "Hello there." | piper -m en_US-lessac-medium -f out.wav')

## 6. Gotchas & Pitfalls

- **Ship *both* files.** A voice is `voice.onnx` **and** `voice.onnx.json`. Copy only the
  `.onnx` and load fails (or you lose the sample rate / `phoneme_id_map`). They always travel
  together.
- **espeak-ng must be reachable.** Phonemization is done by `espeak-ng`. The `piper-tts` wheel
  bundles it, but on stripped containers/embedded images install the system package or the
  phonemizer raises at run time.
- **Sample-rate mismatch = chipmunk/slow-mo.** A `medium` voice is 22.05 kHz, `low`/`x_low` are
  16 kHz. Read `sample_rate` from the config (or the WAV header) and resample if your pipeline
  expects something else — don't assume 44.1/48 kHz.
- **Quality is chosen by *voice*, not at run time.** You can't turn a `low` voice into a `high`
  one with parameters; pick the right tier when you download. `length_scale`/`noise_*` only
  tweak speed and variability, not fidelity.
- **`length_scale` is inverted from "speed".** `length_scale > 1` makes speech **slower/longer**;
  `< 1` faster. People expect the opposite — it scales *duration*, not rate.
- **`speaker_id` only applies to multi-speaker voices.** Passing one to a single-speaker voice
  does nothing; passing an out-of-range id to a multi-speaker voice errors. Check `num_speakers`
  in the config.
- **Minimal control surface.** No real SSML, no `<break>`/`<emphasis>`, no emotion. For pauses,
  split text and insert silence yourself; for emphasis, you're mostly out of luck. If you need
  prosody control, Piper is the wrong tool.
- **Pronunciation is espeak-ng's, for better and worse.** Odd names, acronyms, and numbers are
  pronounced by espeak rules. Normalize/spell-out tricky tokens upstream; there's no learned
  text front end to save you.
- **API churn (piper1-gpl).** The 2025 rewrite changed the Python API (`synthesize_wav` /
  `SynthesisConfig` replaced `synthesize_stream_raw` and loose kwargs). Match snippets to your
  installed version.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs Piper |
|---|---|---|
| **Piper** | Fast, tiny, **offline** TTS on CPU / Raspberry Pi / edge; Home Assistant & Rhasspy | Lower naturalness; **no voice cloning**; minimal prosody/SSML control |
| **Coqui TTS (VITS/XTTS-v2)** | Self-hosted **voice cloning**, training your own voice, multilingual | Much heavier deps, GPU-hungry for XTTS, slower on CPU; community-maintained |
| **StyleTTS2 / Tortoise** | Top open-model expressiveness / naturalness | Far slower and heavier; not real-time on CPU; more setup |
| **ElevenLabs** | Best-in-class naturalness + instant cloning, zero setup | Paid per-character, cloud-only, audio leaves your box |
| **Azure / Google / Amazon Polly** | Managed scale, rich **SSML**, SLAs, huge voice catalog | Per-use cost, network dependency, vendor lock-in |
| **espeak-ng (alone)** | Ultra-light, fully deterministic, every language, accessibility | Robotic/formant — clearly synthetic; Piper *uses* it only for phonemes |

**Rule of thumb:** reach for **Piper** when you need **real-time, offline speech on modest
hardware** and "good enough, natural-ish" quality — assistants, kiosks, accessibility, anything
that must work without a network or a GPU. Step up to **Coqui/XTTS** when you need cloning or
custom-trained voices, to **StyleTTS2/Tortoise** for maximum open expressiveness, and to a
**cloud vendor** when you want the highest naturalness with rich SSML and the least ops.

## 8. Resources

- **Piper (current, piper1-gpl) — GitHub:** install, Python API, CLI, training — https://github.com/OHF-Voice/piper1-gpl
- **Original Rhasspy Piper repo (archived, still-referenced):** https://github.com/rhasspy/piper
- **Voice catalog (`rhasspy/piper-voices` on Hugging Face):** 80+ languages, all qualities — https://huggingface.co/rhasspy/piper-voices
- **Voice samples (listen before you download):** https://rhasspy.github.io/piper-samples/
- **Home Assistant — Piper add-on (real-world deployment):** https://www.home-assistant.io/voice_control/voice_remote_local_assistant/
- **VITS paper — the end-to-end architecture each voice is built on:** https://arxiv.org/abs/2106.06103
- **espeak-ng — the phonemizer Piper relies on:** https://github.com/espeak-ng/espeak-ng

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
PAD, BOS, EOS = "_", "^", "$"


def phonemes_to_ids(phonemes, phoneme_id_map):
    ...


def to_pcm16(samples):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE